In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv('datasets/database_processed_h3.csv')

In [5]:
df_agg = df_raw.groupby(['h3', 'DATA', 'MES_BO', 'ANO_BO']).agg(
    n_furtos = ('FURTO', 'sum'),
    n_roubos = ('ROUBO', 'sum'),
    n_total = ('cnt_crime', 'sum')
).reset_index()

In [6]:
def risco_log(d, col, q=0.99):
    lg = np.log1p(d[col])                                           
    teto = lg.groupby(d["periodo"]).transform(lambda s: s.quantile(q))
    return (lg / teto.replace(0, np.nan)).clip(0, 1).fillna(0) 

In [7]:
df_agg['periodo'] = pd.to_datetime(dict(year=df_agg["ANO_BO"], month=df_agg["MES_BO"], day=1))

In [8]:
train_val = df_agg[(df_agg["periodo"] >= "2022-01-01") & (df_agg["periodo"] <= "2025-06-01")].copy()
test = df_agg[(df_agg["periodo"] >= "2025-07-01") & (df_agg["periodo"] <= "2026-06-01")].copy()

In [9]:
for d in (train_val, test):
    d["risco_furto"] = risco_log(d, "n_furtos")
    d["risco_roubo"] = risco_log(d, "n_roubos")

In [11]:
print("treino+val:", train_val["periodo"].min().date(), "->", train_val["periodo"].max().date(), f"({len(train_val):,} linhas)", f"({round(len(train_val)/len(df_agg)*100,2)}%)")
print("teste     :", test["periodo"].min().date(), "->", test["periodo"].max().date(), f"({len(test):,} linhas)", f"({round(len(test)/len(df_agg)*100,2)}%)")

treino+val: 2022-01-01 -> 2025-06-01 (51,339 linhas) (78.07%)
teste     : 2025-07-01 -> 2026-06-01 (14,419 linhas) (21.93%)


In [12]:
train_val.head()

,h3,DATA,MES_BO,ANO_BO,n_furtos,n_roubos,n_total,periodo,risco_furto,risco_roubo
1,87812e672ffffff,2-9-2024,9,2024,1,0,1,2024-09-01,0.128008,0.000000
2,87a8022d2ffffff,1-5-2024,5,2024,0,1,1,2024-05-01,0.000000,0.161974
5,87a806c15ffffff,1-11-2022,11,2022,0,1,1,2022-11-01,0.000000,0.157827
6,87a806c15ffffff,1-3-2024,3,2024,0,1,1,2024-03-01,0.000000,0.159479
7,87a806c15ffffff,1-5-2024,5,2024,1,0,1,2024-05-01,0.130911,0.000000


validação e busca por hiperparametros do modelo Baseline